### SFS

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
 =============================================
# Model Configuration & Feature Selection Setup
# =============================================
# Initialize LightGBM classifier with binary classification settings
lgbm_model = LGBMClassifier(
    objective='binary',        # Binary classification task
    boosting_type='gbdt',      # Gradient Boosting Decision Tree
    learning_rate=0.05,        # Shrinkage rate for updates
    num_leaves=10,             # Maximum number of leaves per tree
    n_estimators=50,           # Number of boosting rounds
    random_state=42,           # Seed for reproducibility
    n_jobs=-1,                 # Use all available cores
    eval_metric='logloss',     # Evaluation metric during training
    verbose=-1                 # Silence output
)

# Configure sequential feature selection
feature_selector = SFS(
    estimator=lgbm_model,
    k_features=(5, 25),        # Target feature range (min, max)
    forward=True,               # Forward selection approach
    floating=False,             # No floating selection
    scoring='f1',               # Optimization metric
    cv=5,                       # 5-fold cross-validation
    n_jobs=-1,                  # Use all available cores
    verbose=2                   # Medium verbosity
)

# ======================
# Data Preparation
# ======================
# Prepare training data (exclude non-feature columns)
X_train = data_train.drop(columns=['label_binary', 'n_image', 'label_multi'])
y_train = data_train['label_binary']

# Encode target labels (0: no corrosion, 1: corrosion)
label_mapping = {'no corrosion': 0, 'corrosion': 1}
y_train_encoded = y_train.map(label_mapping)

# ======================
# Feature Selection
# ======================
# Perform feature selection using training data
feature_selector.fit(X_train, y_train_encoded)

# ======================
# Results Extraction
# ======================
# Get selected feature names and convert to list
selected_features = list(feature_selector.k_feature_names_)
print("Selected features:", selected_features)

In [ ]:
wrapper_features=['label_binary', 'n_image', 'label_multi']+selected_features
data_train_wrapper=data_train[wrapper_features]
data_test_wrapper=data_test[wrapper_features]